In [351]:
import os
import json
import copy
import requests
import pandas as pd
import numpy as np
import pickle as pkl
from tqdm import tqdm
from random import sample, choice, shuffle
import scipy
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.linear_model import SGDClassifier
# from custom_models.custom_model import BatchVectorizerClassifierCitations
import xgboost as xgb
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn import metrics
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
import importlib

import open_alex_utils

importlib.reload(open_alex_utils)

from open_alex_utils import get_pos_citation_dataset, \
    get_neg_citation_dataset, extract_bow_feats, get_bow_inputs_labels

In [45]:
file_list = [file for file in os.listdir('open_alex_citations') if 'open' in file]

In [46]:
ids_file = '/home/k2141255/study_design_classifier/data/cleaned_pos_neg_train_val_test_dataset_ids.pkl'

In [153]:
study_design = 'is_guideline'

In [154]:
consort_pmids = [
    20335313,
    20334633,
    20332509,
    20346629,
    20410783,
    21686296,
    20352064,
    20334632,
]

In [155]:
consort_openalex_ids = []
for pmid in tqdm(consort_pmids):
    response = requests.get(f"http://api.openalex.org/works/pmid:{pmid}")
    consort_openalex_ids.append(response.json()['id'])

100%|█████████████████████████████████████████████| 8/8 [00:01<00:00,  4.42it/s]


In [51]:
consort_openalex_ids

['https://openalex.org/W2049103906',
 'https://openalex.org/W2108696783',
 'https://openalex.org/W4236805817',
 'https://openalex.org/W2571766406',
 'https://openalex.org/W2007572208',
 'https://openalex.org/W1674684315',
 'https://openalex.org/W2008023699',
 'https://openalex.org/W2884313467']

In [157]:
with open(ids_file, 'rb') as f:
    prop_typ_id_dict= pkl.load(f)

In [158]:
pos_ids = prop_typ_id_dict[study_design]['pos']
neg_ids = prop_typ_id_dict[study_design]['neg']

In [261]:
pos_path = "../data/pubmed_articles_pos_human.csv"

with open(pos_path, 'r') as f:
    pos_data = pd.read_csv(f)

In [262]:
pos_train_pmids = [pos_data.loc[pos_id]['pmid'] for pos_id in pos_ids['train']]
pos_val_pmids = [pos_data.loc[pos_id]['pmid'] for pos_id in pos_ids['val']]
pos_test_pmids = [pos_data.loc[pos_id]['pmid'] for pos_id in pos_ids['test']]

In [263]:
del pos_data

In [183]:
response = requests.get(f"http://api.openalex.org/works/pmid:{pos_id}")

In [184]:
consort_openalex_ids

['https://openalex.org/W2049103906',
 'https://openalex.org/W2108696783',
 'https://openalex.org/W4236805817',
 'https://openalex.org/W2571766406',
 'https://openalex.org/W2007572208',
 'https://openalex.org/W1674684315',
 'https://openalex.org/W2008023699',
 'https://openalex.org/W2884313467']

In [303]:
pos_train_set, pos_train_feat_lst = get_pos_citation_dataset(pos_train_pmids)
pos_val_set, pos_val_feat_lst = get_pos_citation_dataset(pos_val_pmids)
pos_test_set, pos_test_feat_lst = get_pos_citation_dataset(pos_test_pmids)

100%|████████████████████████████████████| 1370/1370 [00:00<00:00, 63240.92it/s]


In [302]:
len(pos_train_set)

10480

In [318]:
feat_lst = pos_train_feat_lst + pos_val_feat_lst + pos_test_feat_lst

In [306]:
len(neg_train_set)

1

In [264]:
neg_path = "../data/pubmed_articles_neg_human.csv"

with open(neg_path, 'r') as f:
    neg_data = pd.read_csv(f)

In [265]:
neg_train_pmids = [neg_data.loc[neg_id]['pmid'] for neg_id in neg_ids['train']]
neg_val_pmids = [neg_data.loc[neg_id]['pmid'] for neg_id in neg_ids['val']]
neg_test_pmids = [neg_data.loc[neg_id]['pmid'] for neg_id in neg_ids['test']]

In [266]:
del neg_data

In [ ]:
neg_train_set, neg_train_feat_lst = get_neg_citation_dataset(neg_train_pmids, len(pos_train_set))
neg_val_set, neg_val_feat_lst = get_neg_citation_dataset(neg_val_pmids)
neg_test_set, neg_test_feat_lst = get_neg_citation_dataset(neg_test_pmids)

In [320]:
feat_lst += neg_train_feat_lst + neg_val_feat_lst + neg_test_feat_lst

In [315]:
len(neg_train_set)

10480

In [193]:
len(pos_train_set)

10480

In [194]:
pos_ref_train_dir = "pos_train_refs.json"

with open(pos_ref_train_dir, 'w') as f_pos:
    json.dump(pos_train_set, f_pos)

In [195]:
neg_ref_train_dir = "neg_train_refs.json"

with open(neg_ref_train_dir, 'w') as f_neg:
    json.dump(neg_train_set, f_neg)

In [321]:
feat_lst = list(set(feat_lst))

In [322]:
len(feat_lst)

7633222

In [323]:
from collections import defaultdict

freqs = defaultdict(int)

for paper in tqdm(pos_train_set):
    for opid in paper:
        freqs[opid] +=1

for paper in tqdm(neg_train_set):
    for opid in paper:
        freqs[opid] +=1

100%|█████████████████████████████████| 10480/10480 [00:00<00:00, 215897.53it/s]


In [324]:
freqs = {k: v for k, v in sorted(freqs.items(), key=lambda item: item[1], reverse=True)}

In [325]:
freqs

{'https://openalex.org/W2165010366': 244,
 'https://openalex.org/W1247968195': 138,
 'https://openalex.org/W2100837269': 127,
 'https://openalex.org/W1775749144': 105,
 'https://openalex.org/W2093770528': 86,
 'https://openalex.org/W2079065919': 76,
 'https://openalex.org/W2104675861': 73,
 'https://openalex.org/W2135400982': 61,
 'https://openalex.org/W2129688566': 58,
 'https://openalex.org/W2156121263': 58,
 'https://openalex.org/W4293247451': 58,
 'https://openalex.org/W4244959897': 55,
 'https://openalex.org/W1994277369': 54,
 'https://openalex.org/W2148949075': 53,
 'https://openalex.org/W2337454357': 51,
 'https://openalex.org/W2010149513': 51,
 'https://openalex.org/W1980717583': 50,
 'https://openalex.org/W2096074908': 49,
 'https://openalex.org/W2134236426': 47,
 'https://openalex.org/W2158822657': 47,
 'https://openalex.org/W2107277218': 46,
 'https://openalex.org/W2024352468': 44,
 'https://openalex.org/W2769264260': 42,
 'https://openalex.org/W1994158325': 41,
 'https://op

In [326]:
freqs_reduced = {k: v for k, v in freqs.items() if  v > 10}

In [327]:
len(freqs_reduced)

796

In [328]:
feature_lst_reduced = list(freqs_reduced)

In [ ]:
lim_len = -1

In [345]:
bow_neg_train = extract_bow_feats(neg_train_set, feature_lst_reduced)
bow_neg_val = extract_bow_feats(neg_val_set, feature_lst_reduced)
bow_neg_test = extract_bow_feats(neg_test_set, feature_lst_reduced)
bow_pos_train = extract_bow_feats(pos_train_set, feature_lst_reduced)
bow_pos_val = extract_bow_feats(pos_val_set, feature_lst_reduced)
bow_pos_test = extract_bow_feats(pos_test_set, feature_lst_reduced)

100%|█████████████████████████████████████| 1285/1285 [00:00<00:00, 3044.82it/s]


In [353]:
inputs_train, labels_train = get_bow_inputs_labels(bow_pos_train, bow_neg_train, is_shuffle = True)
inputs_val, labels_val = get_bow_inputs_labels(bow_pos_val, bow_neg_val, is_shuffle = True)
inputs_test, labels_test = get_bow_inputs_labels(bow_pos_test, bow_neg_test, is_shuffle = True)

100%|████████████████████████████████████| 2736/2736 [00:00<00:00, 39084.15it/s]


In [358]:
len(labels_train)

20958

In [359]:
len(inputs_train)

20958

In [356]:
class BatchVectorizerClassifierCitations:
    """Custom vectorizer and classifier class to process data in
    batches.  Useful when there are RAM and CPU limitations.

    Args:
        BaseEstimator (Estimator): base class for estimator
    """

    def __init__(
        self,
        loss="hinge",
        penalty="l2",
        alpha=0.0001,
        l1_ratio=0.15,
        fit_intercept=True,
        max_iter=1000,
        tol=0.001,
        shuffle=True,
        verbose=0,
        epsilon=0.1,
        n_jobs=None,
        random_state=None,
        learning_rate="optimal",
        eta0=0.0,
        power_t=0.5,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=5,
        class_weight=None,
        warm_start=False,
        average=False,
    ):
        """intialization of class

        Args:
            clf (Estimator): classifier for training
            vectorizer (Vectorizer): vectorizer to turn text into vectors
            batch_size (int): size of each batch used for training
        """

        self.clf = SGDClassifier(
            loss=loss,
            penalty=penalty,
            alpha=alpha,
            l1_ratio=l1_ratio,
            fit_intercept=fit_intercept,
            max_iter=max_iter,
            tol=tol,
            shuffle=shuffle,
            verbose=verbose,
            epsilon=epsilon,
            n_jobs=n_jobs,
            random_state=random_state,
            learning_rate=learning_rate,
            eta0=eta0,
            power_t=power_t,
            early_stopping=early_stopping,
            validation_fraction=validation_fraction,
            n_iter_no_change=n_iter_no_change,
            class_weight=class_weight,
            warm_start=warm_start,
            average=average,
        )
        self.vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english")
        self.batch_size = 1000
        # self._estimator_type = "classifier"

    def fit(self, X, y):
        """fit the vectorizer and the classifier

        Args:
            X (pd.Series(str)): inputs
            y (np.array(int)): labels
        """

        # check whether vectorizer has been fitted
        vec_fitted = False

        # number of batches to be processed
        if len(y) >= self.batch_size:
            num_batches = len(y) // self.batch_size
        else:
            num_batches = 1

        # Store the classes seen during fit
        self.classes_ = unique_labels(y)

        # for X_chunk, y_chunk in tqdm(
        #     zip(np.array_split(X, num_batches), np.array_split(y, num_batches),),
        #     total=num_batches,
        # ):

        for X_chunk, y_chunk in zip(
            np.array_split(X, num_batches), np.array_split(y, num_batches)
        ):
            self.clf.fit(X_chunk, y_chunk)

        return self

    def predict(self, X):

#         X_unicode = X.values.astype("U")
#         X_trans = self.vectorizer.transform(X_unicode)

        predictions = self.clf.predict(X)
        return predictions

    # def predict_proba(self, X):
    #     # Check if fit has been called
    #     check_is_fitted(self)

    #     # Input validation
    #     X = check_array(X)
    #     X_unicode = X.values.astype("U")
    #     X_trans = self.vectorizer.transform(X_unicode)

    #     proba = self.clf.predict_proba(X_trans)
    #     return proba

    def get_params(self, deep):
        return self.clf.get_params(deep=deep)

    def set_params(self, **parameters):
        return self.clf.set_params(**parameters)

In [360]:
clf = BatchVectorizerClassifierCitations()
clf.fit(inputs_train, labels_train)

In [362]:
preds_train = clf.predict(inputs_train)
preds_val = clf.predict(inputs_val)
preds_test = clf.predict(inputs_test)

In [367]:
def print_metrics(labels, preds):
    print("recall:", metrics.recall_score(labels_train, preds_train))
    print("precision:", metrics.precision_score(labels_train, preds_train))
    print("f1:",  metrics.f1_score(labels_train, preds_train))
    print("accuracy", metrics.accuracy_score(labels_train, preds_train))

In [368]:
print_metrics(labels_train, preds_train)

recall: 0.20125966218150587
precision: 0.8917547568710359
f1: 0.32840236686390534
accuracy 0.5884149250882718


In [369]:
print_metrics(labels_val, preds_val)

recall: 0.20125966218150587
precision: 0.8917547568710359
f1: 0.32840236686390534
accuracy 0.5884149250882718


In [370]:
print_metrics(labels_test, preds_test)

recall: 0.20125966218150587
precision: 0.8917547568710359
f1: 0.32840236686390534
accuracy 0.5884149250882718


In [363]:
recall_train = metrics.recall_score(labels_train, preds_train)
precision_train = metrics.precision_score(labels_train, preds_train)
f1_train = metrics.f1_score(labels_train, preds_train)
accuracy_train = metrics.accuracy_score(labels_train, preds_train)

In [364]:
recall_train

0.20125966218150587

In [365]:
precision_train

0.8917547568710359

In [366]:
f1_train

0.32840236686390534